### Code Commentator

In [ ]:
# Imports
import os
from openai import OpenAI
import gradio as gr
from dotenv import load_dotenv


In [ ]:
# Load env variables
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if openrouter_api_key:
    print("OPENROUTER_API_KEY is set.")
else:
    print("OPENROUTER_API_KEY is not set.")


In [ ]:
# Constants
MODEL_GPT = 'openai/gpt-4o-mini'
MODEL_CLAUDE = 'anthropic/claude-haiku-4.5'
MODEL_GEMINI = 'google/gemini-2.5-flash'
OPENROUTER_URL = "https://openrouter.ai/api/v1"

In [ ]:
openrouter = OpenAI(api_key=openrouter_api_key, base_url=OPENROUTER_URL)

In [ ]:
system_prompt = """You are a code documentation assistant.

Your task is to analyze provided source code, detect the programming language, and add clear, idiomatic documentation comments appropriate for that language.

Rules

- Automatically detect the programming language.

- Use the correct documentation style for that language (e.g., docstrings, JSDoc, Javadoc, GoDoc, Doxygen, etc.).

- Preserve original code exactly — do not refactor, rename, or restructure.

- Focus on intent and behavior, not trivial syntax.

- Keep comments concise and professional.

- Do not over-comment obvious lines.

Output Requirements

- Return the complete documented code.

- Output only a single fenced code block labeled with the detected language.

- Do not include explanations outside the code block.

If the language is ambiguous, respond with failed to detect language and do not return any code.
"""

In [ ]:
# Get user input
def user_prompt(code):
    return f"""Analyze the following source code and add appropriate documentation comments:
{code}
"""

In [ ]:
# generate comments
def generate_comments(code,model):
    response = openrouter.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt(code)}
        ],
        max_tokens=2048,
        temperature=0.2
    )
    return response.choices[0].message.content

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# Code Documentation Assistant")
    with gr.Row():
        with gr.Column():
            code_input = gr.Code(label="Source Code")
        with gr.Column():
            output = gr.Code(label="Documented Code")
    with gr.Row():
        with gr.Column():
            model_selector = gr.Dropdown(label="Select Model", choices=[MODEL_GPT, MODEL_CLAUDE, MODEL_GEMINI], value=MODEL_GPT, interactive=True)
        with gr.Column():
            doc_button = gr.Button("Generate Documentation", variant="primary")
    doc_button.click(fn=generate_comments, inputs=[code_input, model_selector], outputs=output)
demo.launch(inbrowser=True)
    